# Kaggle Housing Prices — ML Pipeline
## Phase 1: Project Setup & Data Ingestion

**Competition:** House Prices: Advanced Regression Techniques  
**Goal:** Predict SalePrice using regression models  
**Stack:** pandas, numpy, scikit-learn, xgboost, matplotlib, seaborn

---
## Cell 1 — Install all dependencies
Run once. After this, restart the kernel and proceed.

In [ ]:
# Install all required packages
# Run this cell once, then restart the kernel

!pip install pandas numpy scikit-learn xgboost matplotlib seaborn missingno scipy kaggle --quiet

print('All packages installed.')

---
## Cell 2 — Download data via Kaggle CLI

**Before running this cell:**
1. Go to https://www.kaggle.com → Account → API → Create New Token
2. This downloads a `kaggle.json` file
3. Place it at `~/.kaggle/kaggle.json` (Linux/Mac) or `C:/Users/<you>/.kaggle/kaggle.json` (Windows)
4. Accept competition rules at: https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/rules

In [ ]:
import os
import zipfile

# ── Step 1: Set correct permissions on kaggle.json (Linux/Mac only)
# Windows users: skip this line
os.system('chmod 600 ~/.kaggle/kaggle.json')

# ── Step 2: Create a data directory
os.makedirs('data', exist_ok=True)

# ── Step 3: Download competition files
os.system('kaggle competitions download -c house-prices-advanced-regression-techniques -p data/')

# ── Step 4: Unzip into data/
zip_path = 'data/house-prices-advanced-regression-techniques.zip'
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('data/')
    print('Unzipped successfully.')
else:
    print('ZIP not found. Check that kaggle.json is set up and rules are accepted.')

# ── Step 5: Confirm files exist
for f in os.listdir('data/'):
    print(f'  data/{f}')

---
## Cell 3 — Import all libraries

We import everything upfront so nothing breaks mid-notebook later.

In [ ]:
# ── Core data libraries
import numpy as np
import pandas as pd

# ── Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import missingno as msno          # Missing value visualisation

# ── Stats
from scipy import stats
from scipy.stats import skew, norm

# ── Scikit-learn: preprocessing
from sklearn.preprocessing import RobustScaler, OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# ── Scikit-learn: models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# ── Scikit-learn: evaluation
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

# ── XGBoost
from xgboost import XGBRegressor

# ── Utilities
import warnings
warnings.filterwarnings('ignore')  # Suppress convergence warnings during tuning

# ── Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

# ── Reproducibility seed
SEED = 42
np.random.seed(SEED)

print('All imports successful.')

---
## Cell 4 — Load data

We load both train.csv and test.csv immediately.
The test set has no SalePrice column — that is what we predict at the end.

In [ ]:
# ── Load both files
train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test.csv')

# ── Save the test IDs now — we'll need them for submission later
test_ids = test['Id']

print(f'Train shape : {train.shape}')   # Expected: (1460, 81)
print(f'Test shape  : {test.shape}')    # Expected: (1459, 80)  — no SalePrice column
print(f'Test IDs saved: {len(test_ids)} rows')

---
## Cell 5 — First look: head()

`head()` shows the first 5 rows. Scroll right — there are 80 columns.
You'll see a mix of numbers (LotArea, YearBuilt) and strings (MSZoning, Street).
The rightmost column in train is SalePrice — our target.

In [ ]:
train.head()

---
## Cell 6 — Column types: dtypes & info()

`info()` tells you:
- How many non-null values each column has (missing = 1460 minus this number)
- The dtype: `object` = string/categorical, `int64`/`float64` = numeric

You'll spot right away that many columns have fewer than 1460 non-null values — those need imputation in Phase 3.

In [ ]:
# Full column audit
train.info(verbose=True, show_counts=True)

---
## Cell 7 — Numeric summary: describe()

`describe()` shows count, mean, std, min, quartiles, max for every numeric column.

Key things to notice:
- **SalePrice**: ranges from 34,900 to 755,000 — large spread, likely skewed
- **LotArea**: max is ~215,245 — far from the 75th percentile of 11,602. Outliers.
- **MasVnrArea, GarageYrBlt**: will show NaN in count — missing values
- **BsmtFinSF2, EnclosedPorch**: mostly zeros — sparse features

In [ ]:
# Transpose so columns are rows — easier to read with 80 features
train.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

---
## Cell 8 — Categorical column audit

Identifies every `object` dtype column, how many unique values it has, and the most common value.

This tells you which columns need One-Hot encoding vs Ordinal encoding (Phase 3).

In [ ]:
cat_cols = train.select_dtypes(include='object').columns.tolist()
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
num_cols.remove('SalePrice')   # Remove target from feature list
num_cols.remove('Id')          # Remove ID — not a feature

print(f'Categorical columns : {len(cat_cols)}')
print(f'Numeric columns     : {len(num_cols)}')
print(f'Total features      : {len(cat_cols) + len(num_cols)}')
print()

# Quick summary of each categorical column
cat_summary = pd.DataFrame({
    'unique_values' : [train[c].nunique() for c in cat_cols],
    'missing'       : [train[c].isnull().sum() for c in cat_cols],
    'top_value'     : [train[c].value_counts().index[0] for c in cat_cols],
    'top_freq'      : [train[c].value_counts().values[0] for c in cat_cols],
}, index=cat_cols).sort_values('missing', ascending=False)

print(cat_summary.to_string())

---
## Cell 9 — Quick missing value count (top 20)

Before full EDA, get a headline picture of which columns need the most attention.

In [ ]:
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print(f'Columns with missing values: {len(missing)} out of {train.shape[1]}')
print()
print(missing.to_string())

---
## Cell 10 — Quick target preview

A fast sanity-check plot of SalePrice distribution before the full EDA in Phase 2.
You will clearly see it is right-skewed — confirming we need the log transform.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Left: raw SalePrice
sns.histplot(train['SalePrice'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('SalePrice — raw (skewed)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].set_xlabel('Sale Price')

# ── Right: log-transformed SalePrice
sns.histplot(np.log1p(train['SalePrice']), kde=True, ax=axes[1], color='darkorange')
axes[1].set_title('log(SalePrice) — normalized')
axes[1].set_xlabel('log(1 + Sale Price)')

plt.suptitle('Target Variable Distribution', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('target_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'SalePrice skewness (raw)         : {train["SalePrice"].skew():.3f}')
print(f'SalePrice skewness (log-transform): {np.log1p(train["SalePrice"]).skew():.3f}')
print()
print('Rule of thumb: |skewness| < 0.5 is roughly symmetric.')
print('The log transform brings us from ~1.88 down to ~0.12 — much better.')

---
## Cell 11 — Phase 1 summary checkpoint

Print a clean summary of everything we know before moving to EDA.

In [ ]:
print('=' * 55)
print('  PHASE 1 CHECKPOINT')
print('=' * 55)
print(f'  Training rows        : {train.shape[0]}')
print(f'  Test rows            : {test.shape[0]}')
print(f'  Total features       : {train.shape[1] - 2}  (excl. Id & SalePrice)')
print(f'  Numeric features     : {len(num_cols)}')
print(f'  Categorical features : {len(cat_cols)}')
print(f'  Cols with NaN (train): {(train.isnull().sum() > 0).sum()}')
print(f'  Target skewness (raw): {train["SalePrice"].skew():.3f}  → needs log transform')
print('=' * 55)
print('  Next: Phase 2 — Full EDA')
print('=' * 55)